# Compare MLflow experiments

Load runs from the **world-cup-scores** MLflow experiment, rank by LOO, and compare models with ArviZ.

In [ ]:
import os
from pathlib import Path

import arviz as az
import mlflow
import pandas as pd
from mlflow.tracking import MlflowClient

PROJECT_ROOT = Path("..").resolve()
EXPERIMENT_NAME = "world-cup-scores"
mlflow.set_tracking_uri(
    os.environ.get("MLFLOW_TRACKING_URI", f"sqlite:///{PROJECT_ROOT / 'mlflow.db'}")
)
client = MlflowClient()

In [ ]:
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    raise RuntimeError(
        f"No experiment '{EXPERIMENT_NAME}'. Run: python -m src.experiments experiments/configs/quick_test.yaml"
    )

runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.loo_elpd DESC"],
)

rows = []
for run in runs:
    m = run.data.metrics
    p = run.data.params
    rows.append({
        "run_id": run.info.run_id,
        "run_name": run.info.run_name,
        "status": run.info.status,
        "loo_elpd": m.get("loo_elpd"),
        "loo_se": m.get("loo_se"),
        "loo_p": m.get("loo_p"),
        "max_rhat": m.get("max_rhat"),
        "divergences": m.get("divergences"),
        "n_matches": m.get("n_matches"),
        "model.home_advantage": p.get("model.home_advantage"),
        "data.years": p.get("data.years"),
    })

runs_df = pd.DataFrame(rows)
runs_df

In [ ]:
def download_trace(run_id: str) -> Path:
    run_dir = PROJECT_ROOT / "experiments" / "runs" / run_id
    trace_path = run_dir / "trace.nc"
    if not trace_path.exists():
        client.download_artifacts(run_id, "trace.nc", dst_path=str(run_dir))
        trace_path = run_dir / "trace.nc"
    return trace_path


def load_traces_for_runs(run_ids: list[str]) -> dict[str, object]:
    compare_dict = {}
    for run_id in run_ids:
        run = client.get_run(run_id)
        name = run.info.run_name or run_id
        trace_path = download_trace(run_id)
        compare_dict[name] = az.from_netcdf(trace_path)
    return compare_dict


top_n = 5
valid = runs_df.dropna(subset=["loo_elpd"]).head(top_n)
compare_dict = load_traces_for_runs(valid["run_id"].tolist())
print(f"Loaded {len(compare_dict)} runs for comparison")

In [ ]:
if len(compare_dict) >= 2:
    comparison = az.compare(compare_dict, method="stacking")
    display(comparison)
else:
    print("Need at least 2 runs with LOO metrics to compare.")

## Inspect one run

Pick a `run_id` from the table above to reload its trace and inspect parameters.

In [ ]:
if len(valid) > 0:
    best_run_id = valid.iloc[0]["run_id"]
    import arviz_stats as azstats

    idata = az.from_netcdf(download_trace(best_run_id))
    display(azstats.summary(idata))
else:
    print("No runs with LOO available yet.")